# Google Merchandise Store: a month in the life of a shop

In this tutorial we look at the [Google Merchandise Store](https://shop.googlemerchandisestore.com/) – a shop that sells Google-branded goods, whose [anonymised clickstream](https://developers.google.com/analytics/bigquery/web-ecommerce-demo-dataset) Google publishes as a BigQuery demo dataset. It covers a single month, January 2021: 93K users, 109K sessions, 493K events.

Using this real-world dataset, we show how the [retentioneering](https://retentioneering.com/docs) library helps you study user behaviour and find the weak spots in a product.

The funnel overview in section 2 leaves us with two research questions:
- Why do we lose users while they are choosing a product? Half of all sessions reach a product listing page, 7.8% reach a product page, and only 1.3% add anything to the basket. Section 3 digs into this.
- Why does the `basket` → `shipping_details` conversion drop? Checkout loses most of its sessions at the very first step: only 30% get through it, while every later step converts above 70%. Section 4 digs into this.

**What we found**
- The apparel hub page is a poor entry point. 15% of all sessions in the shop start there, yet 88% of them end right away (against 56% for sessions that start on a sub-page of the same section), and they end in a purchase 20 times less often. We show that the page itself is not to blame (when a session does not start there, it converts as well as any other page), nor are the traffic sources (by UTM tags they look like those of other landing pages) – the problem is that people who land on it expect to see something else.
- More than half of the sessions with a basket (54%) never start checkout at all: they reach neither the sign-in page nor the address form. This is the main ceiling on conversion, and it sits outside the order form.
- Inside checkout, the entire drop comes from signing in. 15% of the sessions with a basket reach `sign_in` and stop there without a single purchase. For those who do get through it, signing in is no obstacle: conversion is around 60% both for users who were already signed in and for those who signed in on the spot. Users who have just registered convert noticeably worse – 34.7% – and that is a separate opportunity.

We walk through the reasoning that leads to these conclusions with retentioneering, imitating how an analyst would work on a real product problem.

## 1. Preparing and inspecting the data

- If you need the raw dataset, download [`gms.csv.gz`](https://drive.google.com/file/d/19tAZNl6IROsqXaZlTLWX3WMSIUuDzh-J/view?usp=drive_link)
  and put it next to this notebook. The data schema is described in the [README](https://github.com/retentioneering/retentioneering-tools/tree/master/notebooks/GMS/README.md).
- If you are curious how BigQuery data is adapted for retentioneering, or how exactly we prepared this dataset, open `gms_data_preparation.ipynb`. Exporting from BigQuery requires your own `PROJECT_ID`.

### 1.1 Loading the raw data

In [1]:
import pandas as pd
import retentioneering as rete

print("retentioneering", rete.__version__)

df = pd.read_csv("gms.csv.gz", compression="gzip", sep=";")
df["page_location"] = "https://shop.googlemerchandisestore.com" + df["page_location"]
df.head()

retentioneering 5.2.2


,user_id,session_id,event,timestamp,session_number,device,country,utm_source,utm_medium,utm_campaign,product_type,page_location
0,1.000223e+09,1000223163.8035208_1,main,2021-01-07 18:35:34.089387,1,mobile,Australia,(direct),(none),(direct),NaN,https://shop.googlemerchandisestore.com/
1,1.000223e+09,1000223163.8035208_1,main,2021-01-07 18:35:39.094280,1,mobile,Australia,(direct),(none),(direct),NaN,https://shop.googlemerchandisestore.com/
2,1.000300e+06,1000299.7413851356_1,main,2021-01-20 11:04:59.887247,1,desktop,India,(direct),(none),(direct),NaN,https://shop.googlemerchandisestore.com/
3,1.000300e+06,1000299.7413851356_1,main,2021-01-20 11:05:05.002377,1,desktop,India,(direct),(none),(direct),NaN,https://shop.googlemerchandisestore.com/
4,1.000436e+07,10004358.089772267_1,PLP: Bags,2021-01-08 04:10:12.372423,1,mobile,Russia,google,cpc,<Other>,Bags,https://shop.googlemerchandisestore.com/Google...


### 1.2 Loading the data into an Eventstream

[Eventstream](https://retentioneering.com/docs/eventstream) is the central class of retentioneering.
It is essentially a thin wrapper around a `pandas.DataFrame` that knows what each column means. We declare that through the
[schema](https://retentioneering.com/docs/eventstream#schema):

- **`path_cols`** defines what counts as a single path. Usually these are columns such as `user_id` and `session_id`. Our data has both, so we put both into `path_cols`: `user_id` for behaviour over the whole period and `session_id` for a single visit. The first one becomes the default; every method takes `path_col="session_id"` to switch.
- **`segment_cols`** are categorical labels attached to events that split the eventstream into groups whose behaviour we may want to compare. In this dataset they are `country`, `utm_source`, `utm_medium`, `utm_campaign`, `product_type`.
- **`custom_cols`** are extra fields we will need later. Here it is `page_location`, which comes in very handy in case 1 (see section 3.2).

The `event` and `timestamp` columns already use the default names, so there is no need to mention them in the schema.

In [2]:
SCHEMA = {
    "path_cols": ["user_id", "session_id"],
    "segment_cols": [
        "device", "country", "utm_source", "utm_medium", "utm_campaign", "product_type",
    ],
    "custom_cols": ["page_location"],
}

stream0 = rete.Eventstream(df, schema=SCHEMA)
stream0.df.head()

,user_id,session_id,event,timestamp,device,country,utm_source,utm_medium,utm_campaign,product_type,page_location,event_type,subindex,index
0,1.000300e+06,1000299.7413851356_1,main,2021-01-20 11:04:59.887247,desktop,India,(direct),(none),(direct),NaN,https://shop.googlemerchandisestore.com/,raw,2,1
1,1.000300e+06,1000299.7413851356_1,main,2021-01-20 11:05:05.002377,desktop,India,(direct),(none),(direct),NaN,https://shop.googlemerchandisestore.com/,raw,2,2
2,1.000557e+06,1000557.2911835024_1,main,2021-01-07 12:15:33.038847,mobile,India,<Other>,<Other>,<Other>,NaN,https://shop.googlemerchandisestore.com/,raw,2,1
3,1.000557e+06,1000557.2911835024_1,view_promotion,2021-01-07 12:15:38.549142,mobile,India,<Other>,<Other>,<Other>,NaN,https://shop.googlemerchandisestore.com/,raw,2,2
4,1.000557e+06,1000557.2911835024_1,main,2021-01-07 12:15:38.549142,mobile,India,<Other>,<Other>,<Other>,NaN,https://shop.googlemerchandisestore.com/,raw,2,3


The underlying data is available through the `Eventstream.df` accessor. As you can see, three technical columns have been added to the original data: `event_type`, `index`, `subindex`.

The first method worth calling in any analysis is [`Eventstream.describe()`](https://retentioneering.com/docs/eventstream#describe). It reports the basics of the loaded eventstream: its size, date range, event frequencies, the distribution of path lengths and the segment levels.

In [3]:
overview = stream0.describe(top_events=12)

print(overview["shape"])
print(overview["date_range"])
display(overview["path_stats"]["user_id"])
display(overview["event_frequency"])

{'n_events': 493416, 'n_paths': 92983, 'n_unique_events': 47}
{'min': Timestamp('2021-01-01 00:00:08.136569'), 'max': Timestamp('2021-01-31 23:59:55.412363'), 'span': Timedelta('30 days 23:59:47.275794')}


,length,duration
count,92983.000000,9.298300e+04
mean,5.306518,3.612019e+04
std,10.437298,1.891713e+05
min,1.000000,0.000000e+00
25%,2.000000,4.894917e+00
50%,2.000000,5.045870e+00
75%,5.000000,6.255278e+01
90%,10.000000,3.211639e+03
99%,49.000000,1.129364e+06
max,593.000000,2.614472e+06


,event,count,share
0,main,116795,0.236707
1,PLP: Apparel,65391,0.132527
2,view_promotion,52209,0.105811
3,PDP: Apparel,36969,0.074925
4,PLP: Shop By Brand,25923,0.052538
5,store,23871,0.048379
6,PLP: Lifestyle,21323,0.043215
7,basket,18680,0.037859
8,add_to_cart,15179,0.030763
9,select_item,9949,0.020164


Two things are worth noticing before we go further:

- There are 47 unique events, but most of them – `PLP: Apparel`, `PDP: Bags`, ... – are two page types,
  Product List Page and Product Details Page, repeated across 14 product categories.
- Most paths are very short: the median path length is 2 events.

## 2. An overview of user behaviour

### 2.1 A naive first step

Let us start by building one of the most important retentioneering widgets – the [transition graph](https://retentioneering.com/docs/widgets/transition-graph) – and see that applying it head-on gets us nowhere.

In [4]:
stream0.transition_graph()

There is not much to read in such a graph. Even in [Auto edge filter](https://retentioneering.com/docs/widgets/transition-graph#edgefilter) mode, which keeps the three most frequent outgoing edges per node, 47 unique events add up to a tangle. The [path analysis guide](https://retentioneering.com/docs/path-analysis#when-the-picture-is-unreadable)
lists the usual ways to get a clearer picture; two of them will be enough here.

### 2.2 Collapsing `PLP` and `PDP`

`PLP: Bags` and `PLP: Drinkware` are really the same category listing page shown for different product types. Those types are stored in the `product_type` column, so merging such events into `PLP` loses nothing. The same goes for `PDP`, the product pages. We rename events with the [rename_events](https://retentioneering.com/docs/data-processors/rename-events) data processor. In total, 14 category listing events and 11 product page events turn into two, and the number of unique events drops from 47 to 24.

In [5]:
plp_pdp = {e: e.split(":")[0] for e in stream0.df["event"].unique() if e.startswith(("PLP:", "PDP:"))}

list(plp_pdp.items())[:4]

[('PLP: Clearance', 'PLP'),
 ('PLP: Lifestyle', 'PLP'),
 ('PLP: Apparel', 'PLP'),
 ('PDP: Apparel', 'PDP')]

In [6]:
stream = stream0.rename_events(plp_pdp)
stream.describe()["shape"]

{'n_events': 493416, 'n_paths': 92983, 'n_unique_events': 24}

### 2.3 Killing duplicate views with `collapse_events`

We call repeated events in a path loops, because the transition graph draws them as an edge from an event back into itself. Sometimes a repeated event points at a problem in the CJM, and sometimes the repetition is perfectly natural. Compare a loop on `shipping_details` with a loop on `add_to_cart`: filling in the address twice during checkout most likely signals trouble, while adding several items to the basket in a row is exactly what people do.

Whether to collapse repeated events into one is always a judgement call. Here we are working top-down, and for now the overall picture matters more than the details, so collapsing helps. We do it with the [`Eventstream.collapse_events()`](https://retentioneering.com/docs/data-processors/collapse-events) data processor and its `loops=True` argument.

The second argument of the call is `path_col="session_id"`. Every retentioneering method accepts it. It lets you switch freely between views of a path at different levels: in our case between single sessions and the full paths of users. We choose `path_col="session_id"` because this analysis is about how users converge on a purchase within a visit: the dataset covers a single month and there are only 1.17 sessions per user, so we are unlikely to see behaviour evolving over time. From here on we use `path_col="session_id"` everywhere in this notebook.

In [7]:
stream = stream.collapse_events(loops=True, path_col="session_id")

print(f"{len(stream0.df):,} events -> {len(stream.df):,} after collapsing repeats")

493,416 events -> 325,374 after collapsing repeats


That made our eventstream about a third lighter.

> A filter we deliberately did *not* apply. The usual temptation is to drop very short
> paths, reasoning that "sessions of 1-2 events are noise and cannot be analysed anyway". In
> our case that would be a mistake. In case 1 we show that short sessions are exactly where one of the insights hides (see section 3).

### 2.4 The same graph, now readable

Here is what the transition graph looks like now:

In [8]:
stream.transition_graph(path_col="session_id")

Node positions on the graph are computed automatically, and they usually need a bit of manual work afterwards. To save the graph as a whole, use [`state_file`](https://retentioneering.com/docs/widgets#saving-widget-state). Everything about its state – manually arranged nodes, filters and so on – then survives a notebook restart:

```python
stream.transition_graph(state_file="gms_transition_graph.json")
```

Some structure of the CJM is visible now. To bring it out explicitly, let us highlight a path on the graph using the [views](https://retentioneering.com/docs/widgets/transition-graph#views) mechanism:

In [9]:
happy_path = ["PLP", "PDP", "add_to_cart", "basket", "shipping_details", "payment_details", "purchase"]

stream.transition_graph(
    views=[{"name": "Happy path", "focus": {"type": "path", "nodes": happy_path}}],
    view="Happy path",
    path_col="session_id"
)

Although the path is labelled `Happy path` on the graph, that does not make it typical: the top right corner shows that the probability of walking it (`P(route)`) is below 0.01%. So, much as we would like users to go through it within a single session, most of them arrive without a firm intention to buy. All the more reason to find out what makes them leave.

### 2.5 Exploring the happy path

Let us now measure the conversion between the steps of the happy path with a [funnel](https://retentioneering.com/docs/widgets/funnel). Funnel steps are ordered and closed: a session reaches step N only if it has passed every previous step in order.

For convenience we split the path in two: before checkout and after.

In [10]:
stream.funnel(steps=["PLP", "PDP", "add_to_cart"], path_col="session_id", height=500)

Half of all sessions (54947) reach the product listing page, 7.8% reach a product page, and only 1.3% add anything to the basket. Section 3 takes up the question separately: why do we lose users while they are choosing a product?

Now the second half of the CJM – checkout:

In [11]:
stream.funnel(
    steps=["basket", "shipping_details", "payment_details", "purchase"],
    path_col="session_id",
    height=460,
)

Checkout loses most of its sessions at the very first step: the `basket` → `shipping_details` conversion is only 30%, noticeably lower than at any later step. That becomes our second research question: why does the `basket` → `shipping_details` conversion drop? Section 4 answers it.

## 3. Why do we lose users while they are choosing a product?

### 3.1 Where sessions end

To see which steps precede `path_end`, the handy tools are [Step matrix](https://retentioneering.com/docs/widgets/step-matrix) and [Step Sankey](https://retentioneering.com/docs/widgets/step-sankey) with the `path_pattern="path_end"` argument, which aligns all paths on their last event.

In [12]:
stream.step_matrix(path_pattern="path_end", path_col="session_id", height=360)

We can see straight away that paths break off most often on `PLP` (39%), with `main` a distant second (21%). Note also that `path_start` dominates 2-3 steps before the end, which most likely means that what we are looking at is mostly short paths of the form `path_start` → `PLP` → `path_end`. Let us keep that in mind.

The transition graph can tell us much the same thing. The most convenient way to see which events precede `path_end` is [**Ego view**](https://retentioneering.com/docs/widgets/transition-graph#ego-view) focused on the `path_end` node. In this mode the graph unfolds the neighbourhood of a single event into a Sankey diagram and shows every incoming and outgoing edge, weighted by `proba_in` for incoming edges and `proba_out` for outgoing ones. With the focus on `path_end` only incoming edges show up, and they are exactly the events that end a session.

In [13]:
stream.transition_graph(
    path_col="session_id",
    views=[{
        "name": "Where sessions end",
        "egoNode": "path_end",                             # opens the Ego view dialog when the view is applied
        "focus": {"type": "node", "event": "path_end"},    # and keeps the node focused underneath it
    }],
    view="Where sessions end",
    height=560,
)

So we get the same top of session-ending events that we saw in the step matrix: `PLP` (39%), `main` (21%), `PDP` (16%).

### 3.2 Looking at individual pages

Let us dig into `PLP` and work out why this page so often turns out to be the last one in a session. Earlier we deliberately collapsed every category page into `PLP` to get a readable high-level picture. Now we need the opposite – to go down into specific categories and check whether some of those pages drive more churn than others. This is where the `page_location` column comes in, processed by the [`urls_to_events`](https://retentioneering.com/docs/data-processors/urls-to-events) data processor. It builds event names out of URLs and lets you collapse branches of the URL tree at any level.

Note the order of operations: first we split by URL, then we collapse repeated events, exactly as we did in the previous section.

In [14]:
all_pages = (
    stream0
    .urls_to_events("page_location", nodes=[], keep_full_paths=True)
    .collapse_events(loops=True, path_col="session_id")
)

print(f"{all_pages.df['event'].nunique()} distinct pages")

931 distinct pages


In [15]:
all_pages.df.head()

,user_id,session_id,event,timestamp,device,country,utm_source,utm_medium,utm_campaign,product_type,page_location,event_type,index,subindex
0,1.000223e+09,1000223163.8035208_1,main://,2021-01-07 18:35:34.089387,mobile,Australia,(direct),(none),(direct),NaN,https://shop.googlemerchandisestore.com/,collapsed,1,2
1,1.000300e+06,1000299.7413851356_1,main://,2021-01-20 11:04:59.887247,desktop,India,(direct),(none),(direct),NaN,https://shop.googlemerchandisestore.com/,collapsed,1,2
2,1.000436e+07,10004358.089772267_1,PLP: Bags:/google+redesign/bags/backpacks,2021-01-08 04:10:12.372423,mobile,Russia,google,cpc,<Other>,Bags,https://shop.googlemerchandisestore.com/Google...,collapsed,1,2
3,1.000557e+06,1000557.2911835024_1,main://,2021-01-07 12:15:33.038847,mobile,India,<Other>,<Other>,<Other>,NaN,https://shop.googlemerchandisestore.com/,raw,1,2
4,1.000557e+06,1000557.2911835024_1,view_promotion://,2021-01-07 12:15:38.549142,mobile,India,<Other>,<Other>,<Other>,NaN,https://shop.googlemerchandisestore.com/,raw,2,2


Now let us see how each page relates to `path_end` in terms of `proba_in` and `proba_out`. Instead of drawing a graph we take the numbers directly with the headless method [Eventstream.transition_graph_data()](https://retentioneering.com/docs/widgets/transition-graph#headless-mode). We also add the total number of visits to each page with `Eventstream.get_event_counts()`, to get a sense of how much traffic passes through them.

In [16]:
pd.set_option("display.max_colwidth", 70)  # event names here are URLs, so give them room

# Share of all transitions into path_end that come from each page
arrivals = all_pages.transition_graph_data(edge_weight="proba_in", path_col="session_id")["path_end"]
# Probability of going from each page into path_end
exits = all_pages.transition_graph_data(edge_weight="proba_out", path_col="session_id")["path_end"]
# traffic to each page
visits = pd.Series(all_pages.get_event_counts())

exits = pd.DataFrame({
    "share of all exits": arrivals,
    "visits": visits,
    "exit rate": exits,
}).round(3).sort_values("share of all exits", ascending=False)

display(exits.head(10))
print("Exit rate quantiles:\n", exits["exit rate"].quantile([0.5, 0.75, 0.9]))

,share of all exits,visits,exit rate
main://,0.213,72438.0,0.320
PLP: Apparel:/google+redesign/apparel,0.154,22986.0,0.733
view_promotion://,0.095,44433.0,0.233
PDP: Apparel:/google+redesign/apparel/google+dino+game+tee,0.065,7722.0,0.922
PLP: Shop By Brand:/google+redesign/shop+by+brand/youtube,0.055,9458.0,0.630
store:/store.html,0.049,16470.0,0.325
PLP: Apparel:/google+redesign/apparel/mens,0.021,10719.0,0.212
PLP: Lifestyle:/google+redesign/lifestyle/drinkware,0.020,6193.0,0.359
basket:/basket.html,0.019,12328.0,0.171
PLP: Clearance:/google+redesign/clearance,0.018,8897.0,0.222


Exit rate quantiles:
 0.50    0.1300
0.75    0.2860
0.90    0.5362
Name: exit rate, dtype: float64


So a single page, `Apparel:/google+redesign/apparel`, accounts for 15.4% of all session endings, and the probability that a session ends while on that page is 73.3%. Both numbers are high enough to suspect that something is wrong with the apparel hub `Apparel:/google+redesign/apparel` (for comparison, the inner page `Apparel:/google+redesign/apparel/mens` scores far lower – 2.1% and 21.2% – while carrying traffic of the same order). Let us check that.

Churn numbers that high may simply reflect the fact that this page is a landing page for part of the incoming traffic (remember, we did not filter out short paths). So the next step is to check whether they stay just as high when the page appears further down a path rather than at its very beginning. We also compare them with similar pages – the sub-pages of the `Apparel` hub.

### 3.3 Exit rate split by scenario

To compare the hub with its own sub-pages, the two must first become two different events. The `aggregate_children` option of `urls_to_events` keeps the top-level URL `/google+redesign/apparel` as an event of its own and folds all of its sub-pages into `sub-page`. After that `rename_events` turns the URLs back into readable names.

In [17]:
pages = stream0.urls_to_events(
    "page_location",
    nodes=[{"path": "/google+redesign/apparel", "aggregate_children": True, "name": "sub-page"}],
).collapse_events(loops=True, path_col="session_id")

pages.df["event"].value_counts().head(8)

event
main://                                                      72471
view_promotion://                                            44432
PDP: Apparel:/google+redesign/apparel/sub-page               23449
PLP: Apparel:/google+redesign/apparel                        22986
PLP: Apparel:/google+redesign/apparel/sub-page               19336
store:/store.html                                            16471
basket:/basket.html                                          12328
PLP: Shop By Brand:/google+redesign/shop+by+brand/youtube     9458
Name: count, dtype: int64

In [18]:
SPLIT = {
    "PLP: Apparel:/google+redesign/apparel": "Apparel hub",
    "PLP: Apparel:/google+redesign/apparel/sub-page": "Apparel sub-page",
    "PLP: Shop By Brand:/google+redesign/shop+by+brand/youtube": "YouTube hub",
}

# keep the three splits above; for the rest, strip the URL and restore the plain name
pages = pages.rename_events({e: SPLIT.get(e, e.split(":")[0]) for e in pages.df["event"].unique()})

pages.df["event"].value_counts().head(8)

event
main                72555
PLP                 61900
view_promotion      50629
PDP                 41750
Apparel hub         22986
Apparel sub-page    19336
store               16489
basket              12439
Name: count, dtype: int64

With [`Eventstream.get_conversion_rate()`](https://retentioneering.com/docs/eventstream#get_conversion_rate) we can now measure the conversion from a given event (an anchor) `start_anchor` into `end_anchor="path_end"` happening on the very next step. The `start_anchor` values we care about are the following [patterns](https://retentioneering.com/docs/path-patterns):

- `path_start->PAGE` – the page was the first event of the session;
- `.->PAGE` – some real event preceded the page (`.` deliberately does not match `path_start`).

The `within=1` argument requires `end_anchor="path_end"` to happen exactly one step after the anchor set by `start_anchor`. A list of several `start_anchor` values measures the conversion into `end_anchor` for each of them.

Conversion here is always about paths: the share of paths in which `start_anchor` occurred and was then followed by `end_anchor`.

In [19]:
listing_pages = ["Apparel hub", "Apparel sub-page", "YouTube hub"]
start_events = [{"pattern": f"{lead}->{page}"} for page in listing_pages for lead in ("path_start", ".")]

pages.get_conversion_rate(
    start_anchor=start_events,
    end_anchor="path_end",
    within=1,
    path_col="session_id",
)[["start_anchor", "paths_with_start", "converted", "conversion_rate"]].round(3)

,start_anchor,paths_with_start,converted,conversion_rate
0,path_start->Apparel hub,17392,15292,0.879
1,.->Apparel hub,4340,1351,0.311
2,path_start->Apparel sub-page,2298,1277,0.556
3,.->Apparel sub-page,9835,2331,0.237
4,path_start->YouTube hub,7289,5200,0.713
5,.->YouTube hub,1834,668,0.364


This table gives us several important observations:

1. The conversion from the landing `Apparel hub` (`path_start->Apparel hub`) into `path_end` – its bounce rate, in other words – is even higher than what we saw before: 87.9% instead of 73%. It is higher than for the landing `Apparel sub-page` and `YouTube hub` (55.6% and 71.3%), but the important part is that it is almost three times the bounce rate of the inner `Apparel hub` (`.->Apparel hub`) – 31.1%.
2. The inner `Apparel hub` has roughly the same bounce rate as the inner `Apparel sub-page` and `YouTube hub`.

It starts to look as if the high bounce rate says less about `Apparel hub` itself than about the traffic that lands on it. But let us keep digging.

### 3.4 Funnel progress by landing page

To be thorough, let us also check whether the conversion from `path_start->Apparel hub` stays low not only in terms of ending the session quickly, but also in terms of moving down the funnel. This time we take `PDP`, `basket` and `purchase` as `end_anchor` instead of `path_end`, and we drop the `within=1` condition.

In [20]:
landing_funnel = pages.get_conversion_rate(
    start_anchor=[
        {"pattern": "path_start->Apparel hub"},
        {"pattern": "path_start->Apparel sub-page"},
        {"pattern": ".->Apparel hub"},
        {"pattern": ".->Apparel sub-page"}],
    end_anchor=["PDP", "basket", "purchase"],
    path_col="session_id",
).round(4)

landing_funnel

,start_anchor,end_anchor,paths_with_start,converted,conversion_rate,base_rate,lift
0,path_start->Apparel hub,PDP,17392,495,0.0285,0.2270,0.1254
1,path_start->Apparel hub,basket,17392,230,0.0132,0.0618,0.2139
2,path_start->Apparel hub,purchase,17392,22,0.0013,0.0100,0.1262
3,path_start->Apparel sub-page,PDP,2298,446,0.1941,0.2270,0.8548
4,path_start->Apparel sub-page,basket,2298,227,0.0988,0.0618,1.5979
5,path_start->Apparel sub-page,purchase,2298,59,0.0257,0.0100,2.5622
6,.->Apparel hub,PDP,4340,1281,0.2952,0.2270,1.3000
7,.->Apparel hub,basket,4340,697,0.1606,0.0618,2.5979
8,.->Apparel hub,purchase,4340,172,0.0396,0.0100,3.9551
9,.->Apparel sub-page,PDP,9835,3743,0.3806,0.2270,1.6763


The conversions from `path_start->Apparel hub` into `PDP`, `basket`, `purchase` differ from the matching conversions from `path_start->Apparel sub-page` by 7, 7 and 20 times respectively (0.0285 VS 0.1941, 0.0132 VS 0.0988, 0.0013 VS 0.0257). In other words, the further down the funnel, the wider the gap.

The lift values for `path_start->Apparel hub` are below 1: starting a session on `Apparel hub` makes `PDP`, `basket` and `purchase` 4-8 times less likely, whereas starting on `Apparel sub-page` makes reaching `basket` and `purchase` 1.6 and 2.6 times more likely.

Note in passing that the ratios of lift for `path_start->Apparel hub` VS `path_start->Apparel sub-page` give the very same 7, 7, 20 as the conversion rates do, because by definition, for any events $A_1, A_2, B$:

$$\frac{lift(A_1\rightarrow B)}{lift(A_2\rightarrow B)}=\frac{conversion\_rate(A_1\rightarrow B) / base\_rate}{conversion\_rate(A_2\rightarrow B) / base\_rate}=\frac{conversion\_rate(A_1\rightarrow B)}{conversion\_rate(A_2\rightarrow B)}.$$

As for the same comparison between `.->Apparel hub` and `.->Apparel sub-page`, the differences in conversion are far less dramatic, and the lift for `.->Apparel hub` is above 1 everywhere.

All of this leads us to a hypothesis: the traffic that arrives on `Apparel hub` is of poor quality.

### 3.5 The landing page as a segment

The attribute "did this session start on `Apparel hub`" can be more than a one-off filter – we can record it as a [segment](https://retentioneering.com/docs/segments). The [`Eventstream.add_segment()`](https://retentioneering.com/docs/data-processors/add-segment) data processor with its `sql` argument assigns segment levels to rows however we like. Let us widen the attribute a little and define a segment with three levels, based on the first event of the session: `Apparel hub`, `Apparel sub-page` or `elsewhere` – they go into the `landing` column.

In [21]:
pages = pages.add_segment(
    "landing",
    sql="""
        SELECT CASE FIRST_VALUE(event) OVER (PARTITION BY session_id ORDER BY "index")
            WHEN 'Apparel hub' THEN 'Apparel hub'
            WHEN 'Apparel sub-page' THEN 'Apparel sub-page'
            ELSE 'elsewhere'
        END
        FROM eventstream
    """,
    path_col="session_id"
)

Now we use the [Segment overview](https://retentioneering.com/docs/widgets/segment-overview) widget to compare a set of [metrics](https://retentioneering.com/docs/path-metrics) across the levels of that segment. Besides the conversions into `PDP`, `basket` and `purchase` that we have already studied, we are interested in `length` and `duration`, and in `in_segment_bulk`. The last one matters most here, because it shows whether traffic sources differ between segment levels. Note also that the conversion rates from section 3.4 can be obtained with the `has_event` metric: the `start_anchor` conditions we passed to `get_conversion_rate` have effectively moved into the levels of the `landing` segment, so the share of paths containing `PDP`, `basket` or `purchase` is numerically the same as the conversion into those events.

One more thing to keep in mind: since the segment has three levels, colouring the table rows red and blue carries little meaning – one of the three values is always bright red and another always bright blue.

In [22]:
pages.segment_overview(
    "landing",
    path_col="session_id",
    metrics=[
        {"metric": "length"},
        {"metric": "duration"},
        {"metric": "has_event", "metric_args": {"event": "PDP"}},
        {"metric": "has_event", "metric_args": {"event": "basket"}},
        {"metric": "has_event", "metric_args": {"event": "purchase"}},
        {"metric": "in_segment_bulk", "metric_args": {"segment_name": "utm_source"}},
        {"metric": "in_segment_bulk", "metric_args": {"segment_name": "utm_medium"}},
        {"metric": "in_segment_bulk", "metric_args": {"segment_name": "utm_campaign"}},
        {"metric": "in_segment_bulk", "metric_args": {"segment_name": "device"}},
        {"metric": "in_segment", "metric_args": {"segment_name": "country", "segment_level": ["United States", "India", "Canada", "United Kingdom"]}},
    ],
)

The `in_segment_bulk_<UTM_TYPE>_any_mean` metrics show that the composition of traffic is about the same inside every level of the segment. For nearly all values of `utm_source`, `utm_medium` and `utm_campaign` the shares of sessions within a level rarely differ by more than 3.3 percentage points: the largest gap between `Apparel hub` and elsewhere is for `utm_medium='(data_deleted)'` (`in_segment_bulk_utm_medium_(data deleted)_any_mean` equals 0.012 and 0.045 respectively).

So the hypothesis from the end of 3.4 does not hold: by UTM tags, the traffic arriving on `Apparel hub` is no different. Its dominant source is organic search, which may of course hide patterns of its own – different groups of search queries, for instance.

In [23]:
hub_sessions = pages.df[pages.df["landing"] == "Apparel hub"].drop_duplicates("session_id")

print("Marketing sources for Apparel hub:")
hub_sessions.groupby(["utm_source", "utm_medium"]).size().sort_values(ascending=False).head(5)

Marketing sources for Apparel hub:


utm_source                       utm_medium
google                           organic       5968
(direct)                         (none)        4127
<Other>                          <Other>       2986
                                 referral      1679
shop.googlemerchandisestore.com  referral       929
dtype: int64

And yet the conversions and the path lengths do differ (the `length_mean` and `duration_mean` metrics). Let us dig deeper and see what causes the difference, if not the traffic sources.

### 3.6 Differences in the first steps

A step matrix in [diff mode](https://retentioneering.com/docs/widgets#diff-mode) visualises step-by-step differences between the paths of two segment levels. Blue cells mean the event is more frequent at that step for `Apparel hub`, red ones for `Apparel sub-page`.

In [24]:
pages.step_matrix(
    diff=("landing", "Apparel hub", "Apparel sub-page"),
    path_col="session_id",
    step_window=5,
    height=380
)

In this form the step matrix tells us nothing new: the main difference of paths from `Apparel hub` is that they are shorter, which is why the `path_end` cells are red. To get a more meaningful picture, let us first truncate every path at its second event with [Eventstream.truncate_paths()](https://retentioneering.com/docs/data-processors/truncate-paths). That removes the short paths made of a single event and, at the same time, removes the difference in the first step that we already know about: some go to `Apparel hub`, others to `Apparel sub-page`.

In [25]:
pages\
    .truncate_paths(start_anchor={"pattern": "path_start->.->.", "at": 2}, end_anchor="path_end", path_col="session_id")\
    .step_matrix(
        diff=("landing", "Apparel hub", "Apparel sub-page"),
        path_col="session_id",
        step_window=5,
        height=380
    )

Now the interesting details show up. When a path does not break off immediately, it turns out that users who landed on `Apparel hub` go to search (the `search` and `view_search_results` events) much more often, and into the `Apparel sub-page` pages as well. This suggests that there is something they dislike about this particular landing page: they do not see what they expected, go to search, and then end the session.

Unfortunately the shop at https://shop.googlemerchandisestore.com/ has been completely redesigned since then, so it is hard to say what was wrong with `Apparel hub` in January 2021 and how exactly it differed from the inner `Apparel sub-page` pages. You can look it up in the WayBackMachine ([Google+Redesign/Apparel](https://web.archive.org/web/20210123165920/https://shop.googlemerchandisestore.com/Google+Redesign/Apparel), [Google+Redesign/Apparel/Mens](https://web.archive.org/web/20210304130937/https://shop.googlemerchandisestore.com/Google+Redesign/Apparel/Mens)), but it yields no new hypotheses.

In real life, were we Google's analysts, we could also look at the queries that brought people to these pages, at the search queries typed inside the shop, and so on. For now all we can state is that users who land on `Apparel hub` expect something else from it. Some of them leave at once, others try to search the site and drop off a little later. So the problem is not the nature of the `Apparel hub` page as such, but the way it works as an entry point to the site.

### 3.7 Case 1 summary

Back to the original question: why do we lose users while they are choosing a product?

We started from the fact that sessions break off most often on the product listing: `PLP` is the last event in 39% of sessions. Digging into individual pages, we found that a sizeable part of that churn comes from one page – the apparel hub `/google+redesign/apparel`. We then tested three explanations in turn:

- The page is bad in itself. Not confirmed: when a user reaches the hub from inside a session, it behaves much like its sub-pages – a bounce rate of 31% against 24% and 36% for its neighbours.
- The traffic is bad. Not confirmed either: by UTM tags the traffic to the hub is no different from the traffic to other landing pages, and most of it is organic.
- Expectations are not met. This one looks right: people who arrive on the hub from outside and do not leave at once go to search, and to the sub-pages of the same section, noticeably more often – that is, they look for what they failed to find on the page.

So the problem lies neither with the traffic nor with the page in general, but with the way the hub works as an entry point. A user arrives from search with a specific query in mind, sees a section showcase and leaves: 88% of such sessions end without opening a single other page, against 56% for sessions that start on a sub-page. They end in a purchase 20 times less often.

What the product team can do about it: rethink what a visitor coming from outside sees on the hub, and check which search queries bring them there.

## 4. Conversion losses in checkout

Let us now work out what causes the conversion drop during checkout after the `basket` step that we noticed in section 2.5.

### 4.1 Analysis with step matrix + get_conversion_rate

As a reminder, users reached the basket in 6743 sessions, but only 1092 of those ended in a purchase. The conversion from `basket` into `shipping_details` is 30%, and into a purchase 16%. We call these the baseline conversions and compare everything below against them.

In [26]:
stream.get_conversion_rate(start_anchor="basket", end_anchor=["shipping_details", "purchase"], path_col="session_id")

,start_anchor,end_anchor,paths_with_start,converted,conversion_rate,base_rate,lift
0,basket,shipping_details,6743,2078,0.308171,0.019096,16.137650
1,basket,purchase,6743,1092,0.161946,0.010020,16.161679


Before we go on, note that this on its own does not necessarily signal a problem. It is common knowledge in e-commerce that shoppers use the basket as a bookmark page for saved items, with no intention of buying at all – or at least not during this visit.

#### 4.1.1 What happens after `basket`

Let us use a centred step matrix to see what happens once users open the basket.

In [27]:
stream.step_matrix(path_pattern="basket", path_col="session_id")

It turns out that the most frequent step after `basket` is `sign_in`, at 24%. This transition matters, because unlike the patterns we look at below, it at least does not mean an outright departure from the checkout funnel on the very first step.

The same share – 24% – goes to `path_end`, but those are clearly paths that never meant to end in a purchase. 21% go to PLP, and those paths were not heading for a purchase either, or at least not right now: people opened the basket and went on choosing products. The same applies to `PDP` at 10%. The transition into `shipping_details`, at 5%, means two things:
- these users are already signed in,
- these users intend to buy.

So we have two sub-patterns and two questions that come with them:
- `basket->sign_in`: can asking users to sign in reduce the conversion into a purchase?
- `basket->shipping_details`: is this transition really a marker of the "happy path", that is, does it lead to a high conversion into a purchase?

To answer them quickly we turn to `get_conversion_rate` again:

In [28]:
stream.get_conversion_rate(
    start_anchor=[
        {"pattern": "basket->sign_in"},
        {"pattern": "basket->shipping_details"}],
    end_anchor="purchase",
    path_col="session_id"
)

,start_anchor,end_anchor,paths_with_start,converted,conversion_rate,base_rate,lift
0,basket->sign_in,purchase,2258,681,0.301594,0.01002,30.098176
1,basket->shipping_details,purchase,1589,833,0.524229,0.01002,52.316431


Indeed, `basket->shipping_details` shows a high conversion of 52% – more than three times the baseline – which certainly expresses a strong intention to buy, so this case is clear.

Next to it, `basket->sign_in` converts at 30%, almost half as much, and yet that is still almost twice the baseline. So where do the users who took this route get lost?

#### 4.1.2 Where the conversion after `basket`→`sign_in` is lost

The comparison we just made between the conversions of `basket->shipping_details` and `basket->sign_in` is not quite fair. The first pattern ends on `shipping_details`, which means those users have already moved deeper into the checkout funnel. Let us look at two numbers instead: how many of the `basket->sign_in` sessions reach `shipping_details`, and how many of those go on to `purchase`.

In [29]:
stream.get_conversion_rate(
    start_anchor="basket->sign_in",
    end_anchor="shipping_details",
    path_col="session_id"
)

,start_anchor,end_anchor,paths_with_start,converted,conversion_rate,base_rate,lift
0,basket->sign_in,shipping_details,2258,1397,0.618689,0.019096,32.398161


In [30]:
stream.get_conversion_rate(
    start_anchor=[
        "basket->sign_in->.*->shipping_details",
        "basket->shipping_details"
    ],
    end_anchor="purchase",
    path_col="session_id"
)

,start_anchor,end_anchor,paths_with_start,converted,conversion_rate,base_rate,lift
0,basket->sign_in->.*->shipping_details,purchase,1397,681,0.487473,0.01002,48.648305
1,basket->shipping_details,purchase,1589,833,0.524229,0.01002,52.316431


Now we can see that once users reach `shipping_details` after `basket->sign_in`, their conversion into a purchase is not that different from those who went into `shipping_details` straight after `basket` (48.7% against 52.4%). What is low is the conversion from `basket->sign_in` into `shipping_details` itself: only 61.8%. So the low conversion we are investigating, $CR(\text{basket->sign\_in}, \text{purchase})$, factors like this:

$$
CR(\text{basket->sign\_in}, \text{purchase}) = CR(\text{basket->sign\_in}, \text{shipping\_details}) \cdot CR(\text{basket->sign\_in->.*->shipping\_details}, \text{purchase}) = 0.487473 * 0.618689 = 0.30159
$$

In other words, this branch loses most of its users because they never reach `shipping_details`.

Let us trace why that happens. We build a step matrix on the pattern `basket->sign_in->[^shipping_details]*->path_end` – the sessions where `basket->sign_in` happened and no `shipping_details` followed.

In [31]:
stream.step_matrix(path_pattern="basket->sign_in->[^shipping_details]*->path_end", path_col="session_id")

We can see that after `basket->sign_in` sessions either ended right away on `path_end` (31%), or went into `registration` (19%), or left the checkout flow altogether: `basket` 18%, `PLP` 9%, `store` 7%, `view_promotion` 8%. Let us check how these transitions affected the conversion into `shipping_details`.

In [32]:
stream.get_conversion_rate(
    start_anchor=[
        "basket->sign_in->registration",
        "basket->sign_in->basket",
        "basket->sign_in->PLP",
    ],
    end_anchor=["shipping_details", "purchase"],
    path_col="session_id"
)

,start_anchor,end_anchor,paths_with_start,converted,conversion_rate,base_rate,lift
0,basket->sign_in->registration,shipping_details,795,625,0.786164,0.019096,41.168096
1,basket->sign_in->registration,purchase,795,214,0.269182,0.010020,26.863565
2,basket->sign_in->basket,shipping_details,222,59,0.265766,0.019096,13.917042
3,basket->sign_in->basket,purchase,222,27,0.121622,0.010020,12.137460
4,basket->sign_in->PLP,shipping_details,102,14,0.137255,0.019096,7.187465
5,basket->sign_in->PLP,purchase,102,4,0.039216,0.010020,3.913604


Of all these subsets of paths, only `basket->sign_in->registration` points at an intention to move further down the checkout funnel: its conversion into `shipping_details` is a fairly high 78%, far above the rest. That said, 78% is still a long way from 100%, and the 26.9% conversion from `basket->sign_in->registration` into a purchase does suggest that this branch loses users along the way.

#### 4.1.3 Why does the conversion drop after `basket->sign_in->registration`?

Let us dig further and see what happened after the `basket->sign_in->registration` pattern. Before building a step matrix, though, it is worth asking what a smooth CJM would look like here. Presumably we would want a user to be returned to the checkout flow automatically after registering, and we would expect to see `shipping_details` shortly afterwards. Now let us see what actually happens.

In [33]:
stream.step_matrix(path_pattern="basket->sign_in->registration", path_col="session_id", step_window=7)

It is quite clear that after registering users go to `store`, `view_promotion` and `basket`, and only then move on to `shipping_details`. The `store` event is a kind of alias for the front page, followed by an (automatic?) `view_promotion`. So it looks as though a successful registration redirects the user to the front page, and they have to find their way back to the basket to carry on with the order.

Let us now check how much this detour costs us in purchases. It corresponds to the pattern `basket->sign_in->registration->[store|view_promotion]*->basket`.

In [34]:
stream.get_conversion_rate(
    start_anchor=[
        {"pattern": "basket->sign_in->registration"},
        {"pattern": "basket->sign_in->registration->[store|view_promotion]*->basket"}],
    end_anchor=["shipping_details", "purchase"],
    path_col="session_id"
)

,start_anchor,end_anchor,paths_with_start,converted,conversion_rate,base_rate,lift
0,basket->sign_in->registration,shipping_details,795,625,0.786164,0.019096,41.168096
1,basket->sign_in->registration,purchase,795,214,0.269182,0.010020,26.863565
2,basket->sign_in->registration->[store|view_promotion]*->basket,shipping_details,549,499,0.908925,0.019096,47.596618
3,basket->sign_in->registration->[store|view_promotion]*->basket,purchase,549,180,0.327869,0.010020,32.720292


As we can see, the detour converts at 32.7% into a purchase and at 90.8% into `shipping_details`. That is, we cannot claim it hurts the CJM all that much: of the 795 paths with `basket->sign_in->registration`, 549 come back to `basket` through the redirect to `store` (549 / 795 = 69%), and 499 of those (90.8%) then reach `shipping_details`. From there 32% end in a purchase – about the same level as the conversion of the `basket->sign_in` predicate (30.1%).

So we cannot claim that this rough edge in the CJM ruins the conversion, even though it would still be better to send the user back to the basket – to the point where we interrupted the checkout flow by asking them to sign in and then to register. It is worth noting separately, though, that about 11% of the paths never reach the basket after registering: the conversion from `basket->sign_in->registration` into `basket` is 89%.

In [35]:
stream.get_conversion_rate(
    start_anchor={"pattern": "basket->sign_in->registration"},
    end_anchor="basket",
    path_col="session_id"
)

,start_anchor,end_anchor,paths_with_start,converted,conversion_rate,base_rate,lift
0,basket->sign_in->registration,basket,795,710,0.893082,0.061818,14.446919


#### 4.1.4 The contribution of behavioural branches to the conversion

Let us pull the previous findings together, split all the paths after `basket` into several mutually exclusive types and measure the conversion and the size of each.

1. Left without starting checkout.
2. Went into `shipping_details` without `sign_in` (already signed in).
3. Went into `shipping_details` after `sign_in`, without registering (had an account but were not signed in).
4. Went into `shipping_details` after registering.
5. Never reached `shipping_details` after registering.
6. Got lost after `sign_in`.

To the usual output of `get_conversion_rate` we add the following columns:
- share – the share of the pattern among all paths that contain `basket`;
- contribution – the share of the `converted` sessions among all paths that contain `basket`;
- magnitude – how far the pattern's conversion into `purchase` deviates from the baseline conversion (in percentage points), weighted by `share`.

`contribution` and `magnitude` are two ways of splitting the baseline conversion across the patterns. The essential difference is that the first one cannot tell apart patterns with zero conversion, however large they are, while the second accounts for both the conversion and the size of a pattern, so `magnitude` is what we will read.

In [36]:
FIRST = "path_start->[^basket]*->basket"   # pins the pattern to the session's first basket
G = "[^sign_in|shipping_details]*"         # no sign-in and no address step on the way
H = "[^shipping_details|registration]*"    # after the sign-in, no registration and no address step
K = "[^shipping_details]*"                 # after the registration, no address step

branches = {
    "1 - left without starting checkout":                     f"{FIRST}->{G}->path_end",
    "2 - straight to shipping_details (already signed in)":   f"{FIRST}->{G}->shipping_details",
    "3 - sign-in to an existing account -> shipping_details": f"{FIRST}->{G}->sign_in->{H}->shipping_details",
    "4 - sign-in -> registration -> shipping_details":        f"{FIRST}->{G}->sign_in->{H}->registration->{K}->shipping_details",
    "5 - sign-in -> registration -> drop-off":                f"{FIRST}->{G}->sign_in->{H}->registration->{K}->path_end",
    "6 - sign-in -> drop-off":                                f"{FIRST}->{G}->sign_in->{H}->path_end",
}

res = stream.get_conversion_rate(
    start_anchor=[{"pattern": p} for p in branches.values()],
    end_anchor="purchase",
    path_col="session_id",
)\
.drop(columns=["start_anchor", "end_anchor", "base_rate", "lift"])
res.insert(0, "branch", list(branches))

n, p = res["paths_with_start"].sum(), res["converted"].sum()
cr = p / n
res["share"] = res["paths_with_start"] / n
res["contribution"] = 100 * res["converted"] / n                        # sums to the CR of the whole sample
res["magnitude"] = 100 * res["share"] * (res["conversion_rate"] - cr)   # sums to zero

total = pd.DataFrame([{
    "branch": "total", "paths_with_start": n, "converted": p, "share": 1.0, "conversion_rate": cr,
    "contribution": 100 * cr, "magnitude": res["magnitude"].sum(),
}])

pd.concat([res.sort_values("magnitude"), total])

,branch,paths_with_start,converted,conversion_rate,share,contribution,magnitude
0,1 - left without starting checkout,3642,0,0.000000,0.540116,0.000000,-8.746942
5,6 - sign-in -> drop-off,807,0,0.000000,0.119680,0.000000,-1.938161
4,5 - sign-in -> registration -> drop-off,216,0,0.000000,0.032033,0.000000,-0.518764
3,4 - sign-in -> registration -> shipping_details,641,223,0.347894,0.095062,3.307133,1.767652
1,2 - straight to shipping_details (already signed in),630,377,0.598413,0.093430,5.590983,4.077921
2,3 - sign-in to an existing account -> shipping_details,807,492,0.609665,0.119680,7.296456,5.358295
0,total,6743,1092,0.161946,1.000000,16.194572,0.000000


Among the patterns with zero conversion the strongest one is `1 - left without starting checkout` (54% of the paths). It is the largest group, and inside those sessions there is no sign of an intention to buy: users reach neither `shipping_details` nor the sign-in page. Changes to the checkout process are unlikely to improve anything here.

The patterns `6 - sign-in -> drop-off` (11.9%) and `5 - sign-in -> registration -> drop-off` (3.2%) can be read as an unwillingness to sign in or register just to buy something. It may be worth offering these users a guest checkout and rescuing those 15% of sessions, although even then a high conversion is not to be expected.

Among the positive patterns, 21% in total come from a quick move into `shipping_details` – either straight after `basket` (pattern 2, 9.3%) or with a short detour through `sign_in` (pattern 3, 11.9%), both converting at around 60%. Another 9.5% belong to the weaker pattern `4 - sign-in -> registration -> shipping_details`, which converts at 34.7%.

### 4.2 Looking for the causes through path clustering

You may have noticed that the analysis in the previous section was rather laborious: we kept drilling into branching paths and measuring conversions. As an alternative way of answering the question "why does the conversion drop?", we can cluster the paths that never reached the target event. Cluster analysis gives us a qualitative description of those same paths in terms of their length and their mix of events.

"Never reached the target event" is expressed by the pattern filter `path_start->[^basket]*->basket->[^shipping_details]*->path_end`, which is literally the union of patterns 1, 5 and 6 from section 4.1.4. Note that the `[^basket]*` part is there to exclude paths that visited the basket both with and without a continuation into `shipping_details` – for example `...basket->shipping_details->...->basket->PLP->path_end`.

After that we truncate the paths from `basket` to `path_end`, to cut the noise from whatever happened before `basket`. Finally we run [cluster analysis](https://retentioneering.com/docs/widgets/cluster-analysis) over event counter metrics (`event_count_bulk`).

In [37]:
(
    stream
    .filter_paths({"metric": "matches_pattern", "metric_args": {"pattern": "path_start->[^basket]*->basket->[^shipping_details]*->path_end"}, "op": "=", "value": True}, path_col="session_id")
    .truncate_paths(start_anchor="basket", end_anchor="path_end", path_col="session_id")
    .cluster_analysis(
        features=[{"metric": "event_count_bulk"}],
        overview_metrics=[
            {"metric": "length"},
            {"metric": "event_count_bulk"},
        ],
        method="kmeans",
        method_args={"n_clusters": "2-8"},
        nmf_components="2-8",
        select={"n_clusters": 3, "nmf_components": 3},
        path_col="session_id"
    )
)

The silhouette tab shows that formally the best value of the metric belongs to a split into two clusters. Such a split, however, tells us very little: it merely separates the 82 longest sessions from everything else. So we take the next best split, into three clusters, using the `select` argument. What does it give us to interpret?

The largest cluster (91.6%) consists of short paths (3.244 on average by the `length` metric), and the counters of most events are close to zero. All except these:
- `basket` = 1.218.
- `sign_in` = 0.222
- `store` = 0.186
- `view_promotion` = 0.303
- `main` = 0.17
- `PLP` = 0.501
- `PDP` = 0.267
`basket` needs no explanation: its counter is above 1 because the pattern filter kept only paths that are guaranteed to contain `basket`. The other events are the departures from the checkout path that we have already seen in the step matrix above: either into `PDP`/`PLP`/`main`, or into `sign_in`/`store`/`view_promotion`.

The second largest cluster (6.7%) holds substantial paths with many different events. In essence they all correspond to a user deliberately browsing the shop without meaning to buy right now.

The third cluster (1.7%) holds sessions with a relatively large number of events such as `faq`, `privacy-policy`, `registration`, `return-policy`, `shipping-information`, `terms-of-use`. These users can be called cautious, or attentive: they read a lot of the additional material about buying and delivery.

So, instead of painstakingly working through every branching path with a step matrix, we got a picture that is rougher in numbers but clearer in meaning, together with an estimate of how frequent each pattern is. Of those who reached the basket but not `shipping_details`:
- 91.6% are short paths with no sign of an intention to buy, although about 22% of them did take the first step towards checkout – they reached `sign_in`.
- 6.7% were deliberately browsing products but did not buy during this visit (which probably means we can treat them as warmed-up users in their next session).
- 1.7% are, most likely, held back from moving down the checkout funnel by terms of purchase and delivery that do not suit them.

### 4.3 Case 2 summary

So, why do sessions lose conversion after reaching the basket:

- More than half of the sessions (54%) show no intention of even starting checkout: they reach neither the sign-in page nor the `shipping_details` form. This is the main ceiling on the baseline conversion of 16%, and it lies outside checkout itself.
- Inside checkout the main loss is signing in and registering. Patterns 5 and 6 account for 15% of all sessions with a basket and for exactly zero purchases. This is a candidate for an A/B test of guest checkout.
- Users who have just registered and returned to the checkout flow – the `shipping_details` event – convert into a purchase in 34.7% of sessions, and that is an opportunity for growth. Such scenarios account for 9.5% of sessions, while the comparable conversion for users who already had an account, the level to aim for, is around 60%.

## 5. Conclusion

We have followed the route of a typical product study: from the overall picture to two concrete questions, and from those to numbers a decision can rest on.

We began with a head-on attempt to use the transition graph, which turned out to be unreadable and forced us to prepare the data first: collapse the product pages into `PLP` and `PDP`, remove repeated events and move to the session level. The funnel over the happy path then revealed two bottlenecks, and they became our research questions.

- Why we lose users while they are choosing a product. More than a third of the losses between the listing and the product page come from sessions that start on the apparel hub. Its traffic is ordinary, and inside a session the page works fine – what it fails at is being an entry point.

- Why the `basket` → `shipping_details` conversion drops. More than half of the sessions with a basket (54%) never start checkout at all, and that is the main factor holding the baseline conversion at 16%. Inside checkout the whole drop comes from a single branch: 15% of sessions reach the sign-in page and stop there without a single purchase. Those who do get through it buy equally well – around 60% – except for users who have just registered, where the figure is 34.7%.

An important caveat: all of this is a set of hypotheses for the product team, not proven causes. Proving them properly calls for A/B tests – on the apparel hub as a landing page, on guest checkout, and on returning users to the basket after registration.

Along the way we used many of the retentioneering tools. The data processors [rename_events](https://retentioneering.com/docs/data-processors/rename-events), [collapse_events](https://retentioneering.com/docs/data-processors/collapse-events), [urls_to_events](https://retentioneering.com/docs/data-processors/urls-to-events), [add_segment](https://retentioneering.com/docs/data-processors/add-segment), [truncate_paths](https://retentioneering.com/docs/data-processors/truncate-paths) and [filter_paths](https://retentioneering.com/docs/data-processors/filter-paths) brought the data to the level of detail we needed and singled out the paths, or the parts of paths, we were interested in. The widgets [Transition graph](https://retentioneering.com/docs/widgets/transition-graph), [Step matrix](https://retentioneering.com/docs/widgets/step-matrix), [Funnel](https://retentioneering.com/docs/widgets/funnel), [Segment overview](https://retentioneering.com/docs/widgets/segment-overview) and [Cluster analysis](https://retentioneering.com/docs/widgets/cluster-analysis) helped us see the structure of the paths and compare groups of users. The method [get_conversion_rate](https://retentioneering.com/docs/eventstream#get_conversion_rate) proved especially handy, letting us measure the conversion of whatever [path sub-patterns](https://retentioneering.com/docs/path-patterns) we cared about into a purchase or any other event.

We hope this tutorial has given you an appetite for analysing user behaviour, and that you will put the library to work on problems of your own.